<a href="https://colab.research.google.com/github/wrbryan/conflux-duck/blob/main/colab_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ConfluxDuck Colab demo
This Colab-ready notebook demonstrates the typical ConfluxDuck workflow in Google Colab: clone the repo, install dependencies, mount Google Drive for persistence, load CSVs into a Drive-backed DuckDB, run relationship inference, review suggestions interactively (or use a safe DataFrame fallback), generate a quick chart, and export query results to Google Sheets using user OAuth.

Run the cells in order. Persist files (unified.duckdb, relationships.json) to Drive so they survive session restarts.


In [1]:
# 1) Clone the repo and install runtime dependencies
!git clone https://github.com/wrbryan/conflux-duck.git || true
%cd conflux-duck
!pip install -r requirements.txt --quiet

Cloning into 'conflux-duck'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 27 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 11.03 KiB | 3.68 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/conflux-duck
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 141.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 152.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 113.5 MB/s eta 0:00:00


In [2]:
# 2) Mount Google Drive and set workspace paths
from google.colab import drive
drive.mount('/content/drive')
import os
WORKDIR = '/content/drive/MyDrive/conflux-duck-work'
os.makedirs(WORKDIR, exist_ok=True)
DB_PATH = f"{WORKDIR}/unified.duckdb"
REL_PATH = f"{WORKDIR}/relationships.json"
print('DB_PATH =', DB_PATH)


Mounted at /content/drive
DB_PATH = /content/drive/MyDrive/conflux-duck-work/unified.duckdb


## 3) Load CSVs into a Drive-backed DuckDB
This will create (or overwrite) the named tables in the DB. The example repo includes a small `data/transactions.csv` you can use to follow along.


In [3]:
!python scripts/load_csvs.py --data-dir data --out-db "$DB_PATH"

Loading data/transactions.csv -> table transactions
Traceback (most recent call last):
  File "/content/conflux-duck/scripts/load_csvs.py", line 50, in <module>
    main(args.data_dir, args.out_db, args.overwrite)
  File "/content/conflux-duck/scripts/load_csvs.py", line 39, in main
    con.execute("INSERT INTO __table_schemas VALUES(?, ?)", [s["table"], str(s["columns"])])
_duckdb.ConversionException: Conversion Error: Malformed JSON at byte 1 of input: unexpected character.  Input: "['date', 'description', 'category', 'amount']"


In [4]:
import json # Ensure json module is available for the script
import os

script_path = 'scripts/load_csvs.py'

# Read the original script content
with open(script_path, 'r') as f:
    script_content = f.read()

# Define the string to find and the replacement string
old_string = 'con.execute("INSERT INTO __table_schemas VALUES(?, ?)", [s["table"], str(s["columns"])])'
new_string = 'con.execute("INSERT INTO __table_schemas VALUES(?, ?)", [s["table"], json.dumps(s["columns"])])'

# Perform the replacement
if old_string in script_content:
    modified_content = script_content.replace(old_string, new_string)
    # Write the modified content back to the file
    with open(script_path, 'w') as f:
        f.write(modified_content)
    print(f"Patched '{script_path}' to use json.dumps for column serialization.")
else:
    print(f"Warning: Could not find '{old_string}' in '{script_path}'. Patch might not be needed or already applied.")



Patched 'scripts/load_csvs.py' to use json.dumps for column serialization.


In [5]:
import os

script_path = 'scripts/load_csvs.py'

# Read the original script content
with open(script_path, 'r') as f:
    script_content = f.read()

# Define the string to find and the replacement string
# We want to add 'import json' after other imports. Let's find the last import statement.
# Based on cell 3f124f64, 'import re' is the last import.
old_import_string = 'import re'
new_import_string = 'import re\nimport json' # Add import json on a new line

if old_import_string in script_content:
    # Only replace the first occurrence of old_import_string to avoid unintended changes
    modified_content = script_content.replace(old_import_string, new_import_string, 1)
    # Write the modified content back to the file
    with open(script_path, 'w') as f:
        f.write(modified_content)
    print(f"Added 'import json' to '{script_path}'.")
else:
    print(f"Warning: Could not find '{old_import_string}' in '{script_path}'. Manual insertion of 'import json' might be needed.")

Added 'import json' to 'scripts/load_csvs.py'.


In [6]:
# Verify the content of load_csvs.py after adding 'import json'
!cat scripts/load_csvs.py

#!/usr/bin/env python3
"""
Scan a directory for CSV files and load each into a DuckDB table named by the file (snake_case).
Records a small schema table in the DB: __table_schemas.
"""
import duckdb
import pandas as pd
import argparse
from pathlib import Path
import re
import json


def table_name_from_path(p: Path):
    name = p.stem
    name = re.sub(r'[^0-9a-zA-Z_]', '_', name)
    return name.lower()


def main(data_dir, out_db, overwrite):
    con = duckdb.connect(out_db)
    con.execute("PRAGMA threads=4")
    csvs = list(Path(data_dir).glob("*.csv"))
    schemas = []
    for csv in csvs:
        tbl = table_name_from_path(csv)
        print(f"Loading {csv} -> table {tbl}")
        df = pd.read_csv(csv)
        # Basic normalize: strip column names
        df.columns = [c.strip() for c in df.columns]
        # Write to duckdb (overwrite)
        con.register("tmp_df", df)
        con.execute(f"CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM tmp_df")
        # record schema
       

In [7]:
import os

script_path = 'scripts/load_csvs.py'

# Read the original script content
with open(script_path, 'r') as f:
    script_content = f.read()

# Check if 'import json' is already present
if 'import json' not in script_content:
    # Find a good place to insert it, e.g., after the first few imports
    # or at the top if no other imports exist.
    # For robustness, we can try to insert it after the last existing 'import' statement.
    # Based on the cat output, 'import re' is the last one.
    insertion_point = 'import re'
    if insertion_point in script_content:
        modified_content = script_content.replace(insertion_point, f'{insertion_point}\nimport json', 1)
        with open(script_path, 'w') as f:
            f.write(modified_content)
        print(f"Successfully added 'import json' to '{script_path}'.")
    else:
        # Fallback: add to the beginning if no 'import re' is found (less ideal but works)
        modified_content = 'import json\n' + script_content
        with open(script_path, 'w') as f:
            f.write(modified_content)
        print(f"Successfully added 'import json' to the beginning of '{script_path}'.")
else:
    print(f"'import json' is already present in '{script_path}'. No changes made.")

# Verify the change immediately
!cat {script_path}

'import json' is already present in 'scripts/load_csvs.py'. No changes made.
#!/usr/bin/env python3
"""
Scan a directory for CSV files and load each into a DuckDB table named by the file (snake_case).
Records a small schema table in the DB: __table_schemas.
"""
import duckdb
import pandas as pd
import argparse
from pathlib import Path
import re
import json


def table_name_from_path(p: Path):
    name = p.stem
    name = re.sub(r'[^0-9a-zA-Z_]', '_', name)
    return name.lower()


def main(data_dir, out_db, overwrite):
    con = duckdb.connect(out_db)
    con.execute("PRAGMA threads=4")
    csvs = list(Path(data_dir).glob("*.csv"))
    schemas = []
    for csv in csvs:
        tbl = table_name_from_path(csv)
        print(f"Loading {csv} -> table {tbl}")
        df = pd.read_csv(csv)
        # Basic normalize: strip column names
        df.columns = [c.strip() for c in df.columns]
        # Write to duckdb (overwrite)
        con.register("tmp_df", df)
        con.execute(f"CREATE OR 

In [8]:
# Re-running the command from cell AfOTrN5XZonR after patching the script.
!python scripts/load_csvs.py --data-dir data --out-db "$DB_PATH"

Loading data/transactions.csv -> table transactions
Done.


In [9]:
# Inspect the content of load_csvs.py to verify the string to be replaced
!cat scripts/load_csvs.py

#!/usr/bin/env python3
"""
Scan a directory for CSV files and load each into a DuckDB table named by the file (snake_case).
Records a small schema table in the DB: __table_schemas.
"""
import duckdb
import pandas as pd
import argparse
from pathlib import Path
import re
import json


def table_name_from_path(p: Path):
    name = p.stem
    name = re.sub(r'[^0-9a-zA-Z_]', '_', name)
    return name.lower()


def main(data_dir, out_db, overwrite):
    con = duckdb.connect(out_db)
    con.execute("PRAGMA threads=4")
    csvs = list(Path(data_dir).glob("*.csv"))
    schemas = []
    for csv in csvs:
        tbl = table_name_from_path(csv)
        print(f"Loading {csv} -> table {tbl}")
        df = pd.read_csv(csv)
        # Basic normalize: strip column names
        df.columns = [c.strip() for c in df.columns]
        # Write to duckdb (overwrite)
        con.register("tmp_df", df)
        con.execute(f"CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM tmp_df")
        # record schema
       

## 4) (Optional) Merge other DuckDB files into the unified DB
If you have other .duckdb files on Drive, attach them and copy their tables into the unified DB.


In [10]:
# Example: if you have another DB at /content/drive/MyDrive/other.duckdb, merge it:
# !python scripts/merge_duckdbs.py --out-db "$DB_PATH" --attach /content/drive/MyDrive/other.duckdb

## 5) Run relationship inference
This scans the DB and writes suggestions to relationships.json.


In [11]:
# Note: Relationship inference requires multiple tables to find connections. If only one table is loaded, relationships.json will be empty.
!python scripts/infer_relationships.py --db "$DB_PATH" --out "$REL_PATH"

Wrote suggestions to /content/drive/MyDrive/conflux-duck-work/relationships.json


It seems `relationships.json` is empty. Let's try running the inference again with more verbose output to see why no relationships were suggested.

In [12]:
!python scripts/infer_relationships.py --db "$DB_PATH" --out "$REL_PATH" --debug

usage: infer_relationships.py [-h] [--db DB] [--out OUT]
infer_relationships.py: error: unrecognized arguments: --debug


---
**Debug Comment:** The `infer_relationships.py` script did not find any relationships to suggest because only one table (`transactions`) has been loaded into the database. Relationship inference requires multiple tables to establish connections. The `--debug` argument was also unrecognized, indicating no built-in verbose debugging for the script.

---
**Note to self:** The `relationships.json` file is empty because only `transactions.csv` has been loaded. To infer relationships, I need to provide additional CSV files. When I'm ready to do this, I will place new CSV files into the `data` directory (e.g., `/content/conflux-duck/data/`) and re-run the CSV loading step (cell `da99bcbe`).

--- Debug Comment ---

No relationships are inferred when only one table is present in the database. The `infer_relationships.py` script requires at least two tables to identify potential connections between them.

In [13]:
import json

with open(REL_PATH, 'r') as f:
    relationships_data = json.load(f)

display(relationships_data)

[]

## 6) Interactive review of inferred relationships
Try the ipywidgets-based interactive reviewer first. If widgets are not available or do not render properly in Colab, use the fallback DataFrame-based manual selection shown next.


In [14]:
# Interactive reviewer (may or may not work in Colab depending on the runtime).
try:
    from scripts.review_relationships import review_relationships
    accepted = review_relationships(REL_PATH, db=DB_PATH)
    print('Accepted (in-memory):', len(accepted))
except Exception as e:
    print('Interactive reviewer failed or widgets unavailable:', e)
    print('Run the fallback cell to review suggestions as a DataFrame and accept indices manually.')


Interactive reviewer failed or widgets unavailable: No module named 'scripts.review_relationships'
Run the fallback cell to review suggestions as a DataFrame and accept indices manually.


## 7) Fallback review (DataFrame + manual accept)
If the interactive UI doesn't work, this cell displays the suggested relationships as a DataFrame; after inspecting it, edit `accepted_indices` to the row indices you want to accept and run the cell to write them into the DB.


In [15]:
import json, pandas as pd, duckdb
sugg = json.load(open(REL_PATH))
df = pd.json_normalize(sugg)
df.index.name = 'index'
display(df)
# After inspecting, set accepted_indices to the indices you want to accept (e.g. [0,2])
accepted_indices = []  # << EDIT this list after reviewing
if accepted_indices:
    accepted = df.loc[accepted_indices].to_dict(orient='records')
    con = duckdb.connect(DB_PATH)
    con.execute("CREATE TABLE IF NOT EXISTS __relationships(from_table VARCHAR, from_col VARCHAR, to_table VARCHAR, to_col VARCHAR, reason VARCHAR, score DOUBLE)")
    for r in accepted:
        con.execute("INSERT INTO __relationships VALUES(?, ?, ?, ?, ?, ?)",
                    [r.get('from_table') or r.get('table'), r.get('from_col') or r.get('column'),
                     r.get('to_table'), r.get('to_col'), r.get('reason'), r.get('score')])
    con.close()
    print('Wrote accepted relationships to DB')
else:
    print('No indices selected; no changes written.')


""
index


No indices selected; no changes written.


## 8) Quick reporting & charts
If a `transactions` table exists, produce a small category summary and a bar chart using Plotly.


In [16]:
import duckdb, plotly.express as px
con = duckdb.connect(DB_PATH)
# Check for transactions table
has_transactions = con.execute("SELECT count(*) FROM information_schema.tables WHERE table_name='transactions'").fetchone()[0] > 0
if has_transactions:
    df = con.execute("SELECT category, -SUM(CASE WHEN amount<0 THEN amount ELSE 0 END) AS expenses FROM transactions GROUP BY category").df()
    display(df)
    fig = px.bar(df, x='category', y='expenses', title='Expenses by category')
    fig.show()
else:
    print('No transactions table in DB. Run the loader first.')


,category,expenses
0,Groceries,72.34
1,Income,-0.00
2,Transport,45.80
3,Housing,950.00
4,Shopping,32.10
5,Health,25.00


## 9) Export a query result to Google Sheets (user OAuth)
This example uses Colab's user OAuth flow to avoid storing service-account keys in the notebook. Run the cell and follow the auth prompt.


In [17]:
# Install lightweight Google client libs if needed
!pip install --quiet gspread google-auth
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
# Replace with your query
try:
    df = con.execute("SELECT * FROM transactions LIMIT 100").df()
    sh = gc.create('ConfluxDuck Transactions (Colab)')
    worksheet = sh.get_worksheet(0)
    worksheet.update([df.columns.values.tolist()] + df.values.tolist())
    print('Sheet URL:', sh.url)
except Exception as e:
    print('Export failed or no transactions table:', e)


/usr/lib/python3/dist-packages/blinker/base.py:96: SyntaxWarning:

invalid escape sequence '\*'

/usr/lib/python3/dist-packages/blinker/base.py:174: SyntaxWarning:

invalid escape sequence '\*'

/usr/lib/python3/dist-packages/blinker/base.py:242: SyntaxWarning:

invalid escape sequence '\*'



Sheet URL: https://docs.google.com/spreadsheets/d/1ajoneByP2xSpKvQDHySr1qVWPEHzBTk3cK3OGOAhQ0o


### Example 1: Compare prices from different vendors and historical prices

In [19]:
# Assuming you have 'parts', 'vendors', and 'purchase_history' tables loaded in DuckDB.
# This example will show historical pricing for a specific part from different vendors.
import duckdb

con = duckdb.connect(DB_PATH)

# Query to get historical prices for a specific part across vendors
# (You'd replace 'example_part_id' with an actual part ID from your data)
query = """
SELECT
    p.name AS part_name,
    v.name AS vendor_name,
    ph.purchase_date,
    ph.price
FROM
    purchase_history ph
JOIN
    parts p ON ph.part_id = p.part_id
JOIN
    vendors v ON ph.vendor_id = v.vendor_id
WHERE
    p.part_id = 'example_part_id' -- Replace with an actual part ID
ORDER BY
    ph.purchase_date ASC, v.name ASC
"""

try:
    # Execute the query and fetch results into a pandas DataFrame
    historical_prices_df = con.execute(query).fetchdf()
    display(historical_prices_df)
except duckdb.CatalogException as e:
    print(f"Error: Table not found. Please ensure 'parts', 'vendors', and 'purchase_history' CSVs are loaded. {e}")
finally:
    con.close()


Error: Table not found. Please ensure 'parts', 'vendors', and 'purchase_history' CSVs are loaded. Catalog Error: Table with name purchase_history does not exist!
Did you mean "sqlite_master"?

LINE 8:     purchase_history ph
            ^


### Example 2: Sort parts by equipment type

In [20]:
# Assuming you have a 'parts' table with an 'equipment_type' column.
import duckdb

con = duckdb.connect(DB_PATH)

# Query to get parts sorted by equipment type
query = """
SELECT
    part_id,
    name AS part_name,
    equipment_type
FROM
    parts
ORDER BY
    equipment_type ASC, part_name ASC
"""

try:
    parts_by_equipment_df = con.execute(query).fetchdf()
    display(parts_by_equipment_df)
except duckdb.CatalogException as e:
    print(f"Error: Table not found. Please ensure 'parts' CSV with 'equipment_type' is loaded. {e}")
finally:
    con.close()


Error: Table not found. Please ensure 'parts' CSV with 'equipment_type' is loaded. Catalog Error: Table with name parts does not exist!
Did you mean "pragma_database_list"?

LINE 7:     parts
            ^


### Example 3: Show work by employee in a time frame

In [21]:
# Assuming you have 'work_orders' (with employee_id, start_date, end_date) and 'employees' tables loaded.
import duckdb
import pandas as pd

con = duckdb.connect(DB_PATH)

# Define your desired time frame and employee ID
start_date = '2023-01-01'
end_date = '2023-03-31'
employee_id = 'employee_123' # Replace with an actual employee ID from your data

query = f"""
SELECT
    e.name AS employee_name,
    wo.work_order_id,
    wo.description,
    wo.start_date,
    wo.end_date
FROM
    work_orders wo
JOIN
    employees e ON wo.employee_id = e.employee_id
WHERE
    e.employee_id = '{employee_id}' AND
    wo.start_date >= '{start_date}' AND
    wo.end_date <= '{end_date}'
ORDER BY
    wo.start_date ASC
"""

try:
    employee_work_df = con.execute(query).fetchdf()
    display(employee_work_df)
except duckdb.CatalogException as e:
    print(f"Error: Table not found. Please ensure 'work_orders' and 'employees' CSVs are loaded. {e}")
finally:
    con.close()


Error: Table not found. Please ensure 'work_orders' and 'employees' CSVs are loaded. Catalog Error: Table with name work_orders does not exist!
Did you mean "pg_indexes"?

LINE 9:     work_orders wo
            ^


These examples illustrate how you can craft specific SQL queries against your DuckDB database to extract and analyze data according to your custom needs. Remember, the key is to have the relevant data loaded into your DuckDB first!

---
Notes:
- Persist DB and relationships.json to Drive.
- ipywidgets may not behave consistently in Colab; use the DataFrame fallback when necessary.
- Avoid storing service-account keys in the repo; load them from Drive only if strictly necessary.


In [18]:
import duckdb
import gc

def close_duckdb_connections():
    closed_count = 0
    for obj in gc.get_objects():
        if isinstance(obj, duckdb.DuckDBPyConnection):
            try:
                obj.close()
                closed_count += 1
                print(f"Closed DuckDB connection: {obj}")
            except Exception as e:
                print(f"Error closing DuckDB connection {obj}: {e}")
    if closed_count == 0:
        print("No active DuckDB connections found to close.")
    else:
        print(f"Successfully closed {closed_count} DuckDB connection(s).")

# Call the function to close connections
close_duckdb_connections()

print("Note: For persistent 'IO Error: Could not set lock on file' issues, a full Colab runtime restart (Runtime -> Restart runtime...) is the most reliable solution to clear all lingering locks.")

No active DuckDB connections found to close.
Note: For persistent 'IO Error: Could not set lock on file' issues, a full Colab runtime restart (Runtime -> Restart runtime...) is the most reliable solution to clear all lingering locks.
